In [2]:
# Make sure to install SAM first if you haven't in your current session:
!pip install git+https://github.com/facebookresearch/segment-anything.git

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-0wloaxrt
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-0wloaxrt
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done


In [5]:
!pip install monai nibabel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.5 MB/s eta 0:00:0000:01


In [8]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import nibabel as nib
from torch.utils.data import Dataset, DataLoader
from monai.transforms import Compose, ScaleIntensityRange, Resize, EnsureType


try:
    from segment_anything import sam_model_registry
except ImportError:
    print("Please install SAM using the pip command above.")

# ==========================================
# 1. DATASET & PREPROCESSING PIPELINE
# ==========================================
class Kaggle_BTCV_Liver_Dataset(Dataset):
    def __init__(self, volume_paths, label_paths):
        self.volume_paths = volume_paths
        self.label_paths = label_paths
        self.valid_slices_map = []
        
        print("Scanning BTCV volumes for Liver (Label 6) slices...")
        for vol_idx, (v_path, l_path) in enumerate(zip(volume_paths, label_paths)):
            lab_data = nib.load(l_path).get_fdata()
            depth = lab_data.shape[2]
            
            for slice_idx in range(depth):
                lbl_slice = lab_data[:, :, slice_idx]
                if np.any(lbl_slice == 6):
                    self.valid_slices_map.append({
                        'vol_idx': vol_idx,
                        'slice_idx': slice_idx
                    })
        
        print(f"Total valid liver slices found: {len(self.valid_slices_map)}")

        # Separate transforms to prevent decimal blurring in masks
        self.img_transforms = Compose([
            ScaleIntensityRange(a_min=-200, a_max=400, b_min=0.0, b_max=1.0, clip=True),
            Resize(spatial_size=(384, 384)), 
            EnsureType(dtype=torch.float32)
        ])
        
        self.lbl_transforms = Compose([
            Resize(spatial_size=(384, 384), mode="nearest"), 
            EnsureType(dtype=torch.float32)
        ])

    def __len__(self):
        return len(self.valid_slices_map)

    def __getitem__(self, idx):
        mapping = self.valid_slices_map[idx]
        v_path = self.volume_paths[mapping['vol_idx']]
        l_path = self.label_paths[mapping['vol_idx']]
        slice_idx = mapping['slice_idx']
        
        vol_data = nib.load(v_path).dataobj
        lab_data = nib.load(l_path).dataobj
        
        img_slice = np.array(vol_data[..., slice_idx])
        lbl_slice_raw = np.array(lab_data[..., slice_idx])
        
        # Isolate liver (6 -> 1.0, else 0.0)
        lbl_slice = np.where(lbl_slice_raw == 6, 1.0, 0.0)
        
        img_slice = img_slice[np.newaxis, ...]
        lbl_slice = lbl_slice[np.newaxis, ...]
        
        img_tensor = self.img_transforms(img_slice)
        lbl_tensor = self.lbl_transforms(lbl_slice)
        
        # Convert grayscale to 3-channel for SAM
        img_tensor = img_tensor.repeat(3, 1, 1) 
        
        return img_tensor, lbl_tensor

# ==========================================
# 2. MoSE MODULE 
# ==========================================
class MoSE_Module(nn.Module):
    def __init__(self, embed_dim=256, num_experts=1024, top_k=512):
        super(MoSE_Module, self).__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        
        # Paper Math: h = H/4, w = W/4. For H=384, experts are 96x96.
        self.shape_experts = nn.Parameter(torch.randn(num_experts, 96, 96))
        
        # Gating Network: Upscales the 24x24 SAM embedding by 4x to match the 96x96 shape experts
        self.gating_network = nn.Sequential(
            nn.ConvTranspose2d(in_channels=embed_dim, out_channels=num_experts, kernel_size=4, stride=4)
        )
        
    def forward(self, e_i):
        # G is now [Batch, 1024, 96, 96]
        G = self.gating_network(e_i) 
        
        # Sparse Activation (Top-K)
        topk_values, topk_indices = torch.topk(torch.abs(G), self.top_k, dim=1)
        sparse_G = torch.zeros_like(G)
        sparse_G.scatter_(dim=1, index=topk_indices, src=G.gather(dim=1, index=topk_indices))
        
        # Generate 96x96 Shape Map
        shape_map = torch.sum(sparse_G * self.shape_experts.unsqueeze(0), dim=1)
        shape_prompt = torch.sigmoid(shape_map)
        
        return shape_prompt, sparse_G

# ==========================================
# 3. SAM + MoSE INTEGRATION WRAPPER
# ==========================================
class Medical_SAM_MoSE(nn.Module):
    def __init__(self, sam_checkpoint_path, model_type="vit_b"):
        super().__init__()
        
        print("Loading foundation SAM model...")
        self.sam = sam_model_registry[model_type](checkpoint=sam_checkpoint_path)
        
        # 1. Resize SAM's hardcoded positional embeddings from 64x64 to 24x24
        old_pos_embed = self.sam.image_encoder.pos_embed.data
        old_pos_embed = old_pos_embed.permute(0, 3, 1, 2)
        new_pos_embed = F.interpolate(old_pos_embed, size=(24, 24), mode='bicubic', align_corners=False)
        self.sam.image_encoder.pos_embed = nn.Parameter(new_pos_embed.permute(0, 2, 3, 1))
        
        # 2. Freeze SAM weights
        for param in self.sam.parameters():
            param.requires_grad = False
            
        # Initialize MoSE (Internal 96x96 math)
        self.mose = MoSE_Module(embed_dim=256, num_experts=1024, top_k=512)
        
        # 3. FIX: Spatial Downsampling Adapter
        # Maps the 96x96 shape prompt to the 24x24 image embedding space using SAM's native compression logic
        self.prompt_adapter = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=64, kernel_size=2, stride=2),
            nn.GELU(),
            nn.Conv2d(in_channels=64, out_channels=256, kernel_size=2, stride=2)
        )

    def forward(self, batched_images):
        with torch.no_grad():
            image_embeddings = self.sam.image_encoder(batched_images)
            
        shape_prompt, sparse_weights = self.mose(image_embeddings)
        
        # Adapter outputs [Batch, 256, 24, 24]
        dense_shape_prompt = self.prompt_adapter(shape_prompt.unsqueeze(1))
        
        sparse_embeddings, _ = self.sam.prompt_encoder(
            points=None, boxes=None, masks=None
        )
        
        # FIX: Resize SAM's hardcoded dense positional encoding (4096 tokens) 
        # to match our native 24x24 embedding space (576 tokens)
        image_pe = self.sam.prompt_encoder.get_dense_pe()
        image_pe_resized = F.interpolate(image_pe, size=(24, 24), mode='bilinear', align_corners=False)
        
        low_res_masks, iou_predictions = self.sam.mask_decoder(
            image_embeddings=image_embeddings,
            image_pe=image_pe_resized,  # Inject the mathematically aligned embeddings here
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_shape_prompt,
            multimask_output=False 
        )
        
        return low_res_masks, shape_prompt, sparse_weights

# ==========================================
# 4. INITIALIZATION & DATALOADER SETUP
# ==========================================
if __name__ == "__main__":
    # Ensure these paths match your Kaggle environment exactly
    vol_paths = sorted(glob.glob('/kaggle/input/datasets/nguynhilonguetdhcn/btcv-data-2/01_btcv/images/*.nii'))
    lab_paths = sorted(glob.glob('/kaggle/input/datasets/nguynhilonguetdhcn/btcv-data-2/01_btcv/labels/*.nii'))
    
    train_ds = Kaggle_BTCV_Liver_Dataset(vol_paths, lab_paths)
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)
    
    # Note: To instantiate Medical_SAM_MoSE, you will need to download a SAM checkpoint
    path_to_checkpoint = "/kaggle/input/datasets/juanestebanberon1/sam-vit-b-01ec64-pth/sam_vit_b_01ec64.pth"
    model = Medical_SAM_MoSE(sam_checkpoint_path=path_to_checkpoint)

Scanning BTCV volumes for Liver (Label 6) slices...
Total valid liver slices found: 1542
Loading foundation SAM model...


In [ ]:
import torch.nn.functional as F
from monai.losses import DiceCELoss

# ==========================================
# 5. LOSS FUNCTIONS & OPTIMIZER
# ==========================================
# 1. Dual Segmentation Loss (Dice + Cross Entropy)
# We set sigmoid=True because our model outputs raw logits/unnormalized prompts
seg_loss_fn = DiceCELoss(include_background=False, sigmoid=True, lambda_dice=0.8, lambda_ce=0.2)

# 2. Load-Balancing Regularization (Coefficient of Variation)
def compute_cv_penalty(sparse_weights):
    """
    Prevents routing collapse by penalizing the variance in expert usage.
    sparse_weights shape: [Batch, Num_Experts, H, W]
    """
    # Calculate the mean activation for each of the 1024 experts across the batch & image
    expert_usage = sparse_weights.mean(dim=(0, 2, 3)) 
    
    mean_usage = expert_usage.mean()
    var_usage = expert_usage.var(unbiased=False)
    
    # CV = Variance / (Mean^2). Adding epsilon (1e-8) to prevent division by zero
    cv_loss = var_usage / (mean_usage ** 2 + 1e-8)
    return cv_loss

def compute_l1_penalty(sparse_weights):
    """Warm-up penalty to force gradients through all experts initially."""
    return torch.sum(torch.abs(sparse_weights))

# 3. Optimizer Setup
# We ONLY pass parameters that require gradients (MoSE + SAM Adapters) to save memory
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = torch.optim.AdamW(trainable_params, lr=3e-3, weight_decay=0.1)

# Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# ==========================================
# 6. THE TRAINING ENGINE
# ==========================================
from torch.cuda.amp import autocast, GradScaler

num_epochs = 150
T_warmup = 500  # Iterations before switching to CV penalty
global_step = 0
beta = 1e-2     # Regularization weight

scaler = GradScaler() # Required for Mixed Precision scaling

print(f"Starting Training on {device}...")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        # Mixed Precision Forward Pass
        with autocast():
            # 1. Forward Pass (Updated Unpacking)
            # We now unpack all three variables directly from the model output
            low_res_masks, shape_prompt, sparse_weights = model(images)
            
            # 2. Upscale outputs to match 384x384 Ground Truth
            final_pred_up = F.interpolate(low_res_masks, size=(384, 384), mode="bilinear", align_corners=False)
            shape_prompt_up = F.interpolate(shape_prompt.unsqueeze(1), size=(384, 384), mode="bilinear", align_corners=False)
            
            # 3. Compute Core Segmentation Losses
            loss_seg_final = seg_loss_fn(final_pred_up, labels)
            loss_seg_shape = seg_loss_fn(shape_prompt_up, labels)
            total_seg_loss = loss_seg_final + loss_seg_shape
            
            # 4. Dynamic Routing Regularization (using the unpacked sparse_weights)
            if global_step < T_warmup:
                reg_loss = beta * compute_l1_penalty(sparse_weights)
            else:
                reg_loss = beta * compute_cv_penalty(sparse_weights)
                
            total_loss = total_seg_loss + reg_loss

        # 6. Mixed Precision Backward Pass
        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += total_loss.item()
        global_step += 1
        
    print(f"Epoch [{epoch+1}/{num_epochs}] | Average Loss: {epoch_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), '/kaggle/working/mose_sam_liver.pth')
print("Training complete. Model weights saved.")